In [ ]:
# Core imports
import os
import sys
from pathlib import Path
from datetime import datetime, date
from dotenv import load_dotenv
import pandas as pd
from edgar import Company, set_identity
from typing import Optional, Literal
import asyncio

# Load environment variables
load_dotenv()

# Set SEC identity
sec_identity = os.getenv("SEC_ID")
if not sec_identity:
    raise ValueError("SEC_ID not found in .env file")
set_identity(sec_identity)

In [ ]:
company = Company("AAPL")
# filings = company.get_filings(form="10-K", amendments=False).latest(10)
# # financials = [filing.obj().financials for filing in filings]
# # revenue = [statement.get_revenue() for statement in financials]
# print(financials)

financials = company.get_financials()
income = financials.income_statement()

In [ ]:
print(income)

In [ ]:
from edgar import Company

company = Company("DIS")
financials = company.get_financials()
income = financials.income_statement()
print(income)

In [ ]:
print(company.income_statement())

In [ ]:
print(income)

In [ ]:
import pandas

In [ ]:
us_income_statement = pd.read_csv(".data/us_income_statement.csv")

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, "..")   # path to project root from notebooks/
from historic_fundamentals import get_pe_stats, get_pe_history, get_estimates, get_sector_stats, get_sector_history
pd.set_option('display.max_columns', None)

ticker = "WMT"

ticker_pe_stats = get_pe_stats(ticker)
import matplotlib.pyplot as plt

ticker_history = get_pe_history(ticker)
#print(ticker_history.round(2).to_string())
ticker_history = ticker_history.set_index("month_end_date")
ax = ticker_history[['pe_rolling_5yr_median', 'normalized_pe_5y']].plot(
    secondary_y=['normalized_pe_5y']
)
ax.set_ylabel('normalized PE 5y')
ax.right_ax.set_ylabel('normalized PE 5y')
plt.show()




In [ ]:
print(ticker_pe_stats.round(3).T.to_string())

In [ ]:
industry_history = get_sector_history("industry", "SEMICONDUCTORS")
print(industry_history)
print(get_sector_stats("industry").round(2).to_string())

In [ ]:
ticker_history["fcf_yield"].plot()

In [ ]:
ticker_history.keys()

In [ ]:
ticker_pe_stats.keys()

In [ ]:
filtered_tickers = ticker_pe_stats[ticker_pe_stats['forward_pe'] <= ticker_pe_stats['current_pe']]
filtered_tickers["diff_current_forward"] = (filtered_tickers["forward_pe"] - filtered_tickers["current_pe"]) / filtered_tickers["current_pe"] * 100
filtered_tickers = filtered_tickers.sort_values("diff_current_forward")
df = filtered_tickers[["ticker", "company_name", "current_pe", "forward_pe", "diff_current_forward"]].copy()
df["diff_current_forward"] = df["diff_current_forward"].round(1).astype(str) + '%'
df[["current_pe", "forward_pe"]] = df[["current_pe", "forward_pe"]].round(1)
print(df.to_string())

In [ ]:

ticker_history[["pe_ratio"]].plot()  

In [ ]:
ticker_history.keys()

In [ ]:
ticker_pe_stats.head()

In [ ]:
from av_financials import get_income, get_balance, get_cashflow, get_overview

dvn_balance = get_balance("NVO", period="quarterly")
dvn_income = get_income("NVO", period="quarterly")
from IPython.display import display

display(dvn_balance.T)
display(dvn_income.T)

In [ ]:
dvn_balance = dvn_balance.set_index("fiscal_date_ending")


In [ ]:
dvn_total_assets = dvn_balance.loc["2026-03-31", "total_assets"]
print(dvn_total_assets)

In [ ]:
dvn_income = get_income("DVN", period="quarterly")
dvn_income = dvn_income.set_index("fiscal_date_ending")
dvn_net_income = dvn_income.loc["2026-03-31", "net_income"]
print(dvn_net_income)

In [ ]:
roa = dvn_net_income / dvn_total_assets
print(roa)

In [6]:
import os
#from dotenv import load_dotenv

#load_dotenv(override=True)
alpha_vantage_api_key = os.getenv('ALPHA_VANTAGE_API_KEY')

import requests
import json

ticker = ["NVDA"]

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = f'https://www.alphavantage.co/query?function=EARNINGS_CALL_TRANSCRIPT&symbol={ticker[0]}&quarter=2027Q1&apikey={alpha_vantage_api_key}'
r = requests.get(url)
data = r.json()

print(json.dumps(data, indent=2))

{
  "symbol": "NVDA",
  "quarter": "2027Q1",
  "transcript": []
}


In [4]:
import os
#from dotenv import load_dotenv

#load_dotenv(override=True)
alpha_vantage_api_key = os.getenv('ALPHA_VANTAGE_API_KEY')

import requests
import json

ticker = ['NVDA']

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = f'https://www.alphavantage.co/query?function=EARNINGS_CALL_TRANSCRIPT&symbol={ticker[0]}&apikey={alpha_vantage_api_key}'
r = requests.get(url)
data = r.json()

print(json.dumps(data, indent=2))

{
  "symbol": "NVDA",
  "quarter": "",
  "transcript": []
}


In [ ]:
import os
#from dotenv import load_dotenv

#load_dotenv(override=True)
alpha_vantage_api_key = os.getenv('ALPHA_VANTAGE_API_KEY')

import requests
import json

ticker = ['CBRS']

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = f'https://www.alphavantage.co/query?function=OVERVIEW&symbol={ticker[0]}&apikey={alpha_vantage_api_key}'
r = requests.get(url)
data = r.json()

print(json.dumps(data, indent=2))

In [1]:
import nest_asyncio
nest_asyncio.apply()

from ib_async import IB

ib = IB()
ib.connect("127.0.0.1", 7497, clientId=1)

accounts = ib.managedAccounts()
print(accounts)

paper_account = accounts[0]
print("Using account:", paper_account)

ib.disconnect()

['DUE946835']
Using account: DUE946835


'Disconnecting from 127.0.0.1:7497, 117 B sent in 8 messages, 19.3 kB received in 403 messages, session time 446 ms.'

In [2]:
from ibapi.client import *
from ibapi.wrapper import *
import threading
import time

class TradingApp(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self,self)

    def position(self, account: str, contract: Contract, position: Decimal, avgCost: float):
        print("Position.", "Account:", account, "Contract:", contract, "Position:", position, "Avg cost:", avgCost)
        
    def positionEnd(self):
       print("PositionEnd")
       
def websocket_con():
    app.run()
    
app = TradingApp()      
app.connect("127.0.0.1", 7496, clientId=1)

con_thread = threading.Thread(target=websocket_con, daemon=True)
con_thread.start()
time.sleep(1) 

app.reqPositions()
time.sleep(1)

ModuleNotFoundError: No module named 'ibapi'